# Goodreads Book Trends: Data Preparation

## Overview

This notebook prepares and integrates two Goodreads book datasets for analysis and relational database construction.

The primary dataset consists of Goodreads genre-based book records, which are cleaned, deduplicated, normalized, and transformed into relational tables for books, authors, genres, and their associated relationships.

A second dataset containing popular Science Fiction and Fantasy books is then cleaned and integrated with the primary Goodreads dataset. Matching records are identified using normalized book titles and author names, creating a validated cross-dataset relationship.

The final prepared datasets are saved as Parquet files and are structured to support the SQLite relational database used in the remainder of the project.

## Data Preparation Objectives

This notebook performs the following tasks:

1. Load and inspect the Goodreads Parquet source files.
2. Combine the genre-based source files into a single dataset.
3. Clean publication years, authors, genres, and book identifiers.
4. Create normalized and unique book keys.
5. Build relational DataFrames for books, authors, genres, and relationship tables.
6. Load and clean the separate Science Fiction and Fantasy dataset.
7. Create normalized matching fields for cross-dataset integration.
8. Match corresponding books between the two datasets.
9. Validate the uniqueness and integrity of the cross-dataset relationships.
10. Assign database primary keys and convert relationship tables to foreign keys.
11. Validate the final relational structure.
12. Save all prepared tables as Parquet files for database construction.

## Final Data Structure

The prepared data is organized into the following primary relational tables:

- **BOOKS** — unique books and their Goodreads metadata.
- **AUTHORS** — unique authors.
- **BOOK_AUTHORS** — many-to-many relationships between books and authors.
- **GENRES** — unique Goodreads genres.
- **BOOK_GENRES** — many-to-many relationships between books and genres.
- **SOURCE_GENRES** — the original genre-based source datasets.
- **BOOK_SOURCE_GENRES** — relationships between books and their original source datasets.

The Science Fiction and Fantasy dataset is maintained separately and connected to the primary `BOOKS` table through a validated cross-dataset matching table.

## Output

The completed preparation process produces a set of cleaned, validated Parquet files in `Data/prepared/`. These files provide the finalized inputs for constructing the SQLite relational database and performing SQL-based analysis.

### 1. Import Libraries and Inspect Source Data

The analysis begins by importing the libraries needed for file management, data processing, and Parquet file handling. The source dataset is organized into multiple Parquet files, so the code calculates and displays the size of each file as well as the total dataset size.

This initial inspection helps verify the available source files and provides an understanding of the dataset's overall size before the data is loaded and processed.

In [7]:
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import os
import pandas as pd
import numpy as np


total = 0
folder = Path("../Data/Goodreads_Books/genres_top100")
for f in folder.glob("*.parquet"):
    size = os.path.getsize(f)
    total += size
    print(f"{f.name}: {size / 1e6:.1f} MB")
print(f"Total: {total / 1e6:.1f} MB")

action.parquet: 4.6 MB
adult.parquet: 19.4 MB
adventure.parquet: 13.6 MB
amazon.parquet: 6.0 MB
american_history.parquet: 5.1 MB
animals.parquet: 9.1 MB
anthologies.parquet: 6.8 MB
art.parquet: 15.2 MB
audiobook.parquet: 39.5 MB
bdsm.parquet: 5.3 MB
biography.parquet: 25.0 MB
biography_memoir.parquet: 6.7 MB
book_club.parquet: 7.6 MB
british_literature.parquet: 7.1 MB
business.parquet: 9.7 MB
chick_lit.parquet: 8.4 MB
childrens.parquet: 33.9 MB
christian.parquet: 13.5 MB
christianity.parquet: 5.7 MB
christian_fiction.parquet: 5.2 MB
christmas.parquet: 4.6 MB
classics.parquet: 15.9 MB
comics.parquet: 25.8 MB
comic_book.parquet: 4.5 MB
contemporary.parquet: 41.4 MB
contemporary_romance.parquet: 20.5 MB
cookbooks.parquet: 9.5 MB
cooking.parquet: 5.8 MB
crime.parquet: 16.3 MB
drama.parquet: 5.2 MB
ebooks.parquet: 22.0 MB
economics.parquet: 5.1 MB
education.parquet: 5.6 MB
erotica.parquet: 14.2 MB
essays.parquet: 5.2 MB
family.parquet: 6.2 MB
fantasy.parquet: 56.3 MB
fiction.parquet: 122.9 

### 2. Inspect Source Dataset Structure

A sample Parquet file is loaded to inspect the column names available across the source dataset. This step establishes the structure of the raw data before selecting the fields needed for analysis and helps identify unnecessary columns that can be removed during the cleaning process.

In [8]:
folder = Path("../Data/Goodreads_Books/genres_top100")

# First: check what columns exist and drop anything you don't need
sample = pq.read_table(next(folder.glob("*.parquet")))
print(sample.column_names)

['id', 'name', 'author', 'url', 'genres', 'summary_clean', 'pub_year', 'star_rating', 'num_ratings', 'isbn_clean']


### 3. Combine Genre Datasets

The source data is distributed across multiple Parquet files representing the top books within individual genres. The required columns are selected from each file to keep the dataset focused on information needed for the analysis, including book identifiers, titles, authors, genres, publication years, ratings, and ISBN values.

Each file is also assigned a `genre` column based on its filename so that the original genre classification is preserved after the datasets are combined. The individual tables are then concatenated into a single dataset and saved as `all_genres_combined.parquet` for use in subsequent cleaning and analysis.

In [9]:
folder = Path("../Data/Goodreads_Books/genres_top100")

keep_cols = ["id", "name", "author", "genres", "pub_year", "star_rating", "num_ratings", "isbn_clean"]

tables = []
for file in folder.glob("*.parquet"):
    table = pq.read_table(file, columns=keep_cols)
    genre_col = pa.array([file.stem] * table.num_rows)
    table = table.append_column("genre", genre_col)
    tables.append(table)

combined_table = pa.concat_tables(tables, promote_options="default")
pq.write_table(combined_table, "../Data/Goodreads_Books/all_genres_combined.parquet")

### 4. Review the Combined Data Structure

A sample of the selected columns is converted to a Pandas DataFrame and displayed to verify that the expected fields and values were loaded correctly. Reviewing the first several records provides an initial check of the dataset's structure, content, and data quality before continuing with the cleaning process.

In [10]:
sample = pq.read_table(next(folder.glob("*.parquet")), columns=["id", "name", "author", "genres", "pub_year", "star_rating", "num_ratings", "isbn_clean"])
sample.to_pandas().head(10)

,id,name,author,genres,pub_year,star_rating,num_ratings,isbn_clean
0,615225.Sharpe_s_Devil,Sharpe's Devil,[Bernard Cornwell],"[Historical Fiction, Fiction, Historical, War,...",1992,4.14,8141,9780060932299
1,278794.Blood_Fever,Blood Fever,[Charlie Higson],"[Young Adult, Fiction, Adventure, Mystery, Thr...",2006,4.02,7516,9780786836628
2,23124285-detektiv-conan-vs-kaito-kid,Detektiv Conan vs. Kaito Kid,[Gosho Aoyama],"[Manga, Mystery, Komik, Indonesian Literature,...",2004,4.33,231,9783770476374
3,29280242-marvel-s-captain-america,Marvel's Captain America: Sub Rosa,[David McDonald],"[Marvel, Young Adult, Action, Superheroes]",2016,3.39,74,9781772752014
4,2844643-the-awakening,The Awakening,[Jerry Ahern],"[Post Apocalyptic, Action, Dystopia, Fiction, ...",1984,3.87,305,9780821714782
5,266485.Last_of_the_Breed,Last of the Breed,[Louis L'Amour],"[Fiction, Westerns, Adventure, Historical Fict...",1986,4.28,15161,NaN
6,23906444-terrible-tuesday,Terrible Tuesday,[Don Pendleton],"[Thriller, Fiction, Adventure, Action]",1979,4.02,292,9781497687622
7,3986318-terminal-freeze,Terminal Freeze,[Lincoln Child],"[Thriller, Fiction, Mystery, Horror, Science F...",2008,3.83,19403,9780385515511
8,377612.Eureka_Seven,"Eureka Seven: Psalms of Planets, Vol. 2",[Jinsei Kataoka],"[Manga, Science Fiction, Graphic Novels, Shone...",2005,4.05,268,9781594096914
9,14431469-devil-s-pass,Devil's Pass,[Sigmund Brouwer],"[Adventure, Young Adult, Canada, Fiction, Myst...",2012,3.83,595,9781554699384


### 5. Load and Optimize the Combined Dataset

The genre-level Parquet files are loaded and combined into a single Pandas DataFrame while preserving the original genre from each source file in the `source_genre` column. This provides a consistent genre classification that can be used for later comparisons and analysis.

The publication year is converted to a numeric data type, with invalid values converted to missing values. The `source_genre` column is converted to a categorical data type to reduce memory usage, which is important when working with the large number of records in the combined dataset.

In [11]:
folder = Path("../Data/Goodreads_Books/genres_top100")
keep_cols = ["id", "name", "author", "genres", "pub_year", "star_rating", "num_ratings", "isbn_clean"]

tables = []
for file in folder.glob("*.parquet"):
    table = pq.read_table(file, columns=keep_cols)
    genre_col = pa.array([file.stem] * table.num_rows)
    table = table.append_column("source_genre", genre_col)  # from filename, as backup/comparison
    tables.append(table)

combined_table = pa.concat_tables(tables, promote_options="default")
df = combined_table.to_pandas()

# dtype cleanup — helps memory a lot with this many rows
df["pub_year"] = pd.to_numeric(df["pub_year"], errors="coerce", downcast="integer")
df["source_genre"] = df["source_genre"].astype("category")

### 6. Check Dataset Size and Data Types

The dataset dimensions, memory usage, and column data types are reviewed to confirm that the combined DataFrame was loaded successfully and that the earlier data-type optimizations were applied.

This check is especially important for a large dataset because monitoring memory usage helps identify potential performance issues before additional transformations and analysis are performed.

In [12]:
print(df.shape)
print(df.memory_usage(deep=True).sum() / 1e6, "MB")
df.info()

(4820937, 9)
1700.75812 MB
<class 'pandas.DataFrame'>
RangeIndex: 4820937 entries, 0 to 4820936
Data columns (total 9 columns):
 #   Column        Dtype   
---  ------        -----   
 0   id            str     
 1   name          str     
 2   author        object  
 3   genres        object  
 4   pub_year      int16   
 5   star_rating   float64 
 6   num_ratings   int64   
 7   isbn_clean    str     
 8   source_genre  category
dtypes: category(1), float64(1), int16(1), int64(1), object(2), str(3)
memory usage: 592.1+ MB


### 7. Process and Normalize Goodreads Genres

The `genres` field contains multiple genre values for individual books, so the column is first converted into a standardized list of lowercase, trimmed genre names. The data is then exploded so that each book/genre combination receives its own row.

Missing or blank genre values are removed, along with broad `Fiction`, `Nonfiction`, and `Non-fiction` labels. These general classifications are excluded because the analysis focuses on more specific genre-level patterns.

The cleaned book/genre relationships are then separated into two structures: a `book_genres` table containing unique book-to-genre relationships and a `genres` table containing unique genre names. A year-by-genre summary is also created for exploratory analysis of genre representation over time.

The resulting counts and sample records are displayed to verify that the genre relationships were created correctly before being incorporated into the relational database.

In [ ]:
# Convert the genres column into a clean list of genre names
df["genres_list"] = df["genres"].apply(
    lambda g: [str(x).strip().lower() for x in g]
    if isinstance(g, (list, tuple, np.ndarray))
    else []
)

# Create one row for each book/genre combination
exploded = df.explode("genres_list")

# Remove missing or blank genre values
exploded = exploded[
    exploded["genres_list"].notna() &
    (exploded["genres_list"] != "")
].copy()

# Remove the generic Fiction/Nonfiction labels
# These aren't useful for the more specific genre analysis.
fiction_labels = {
    "fiction",
    "non-fiction",
    "nonfiction"
}

exploded = exploded[
    ~exploded["genres_list"].isin(fiction_labels)
].copy()

# Your unique book identifier is currently called "id".
# Rename it to "book_key" for consistency with the database schema.
exploded = exploded.rename(columns={
    "id": "book_key"
})

# Keep only the information needed to establish the
# relationship between a unique book and its genres.
book_genres = (
    exploded[
        ["book_key", "genres_list"]
    ]
    .drop_duplicates()
    .rename(columns={
        "genres_list": "genre_name"
    })
    .reset_index(drop=True)
)

genres = (
    book_genres[["genre_name"]]
    .drop_duplicates()
    .sort_values("genre_name")
    .reset_index(drop=True)
)

print(f"Book-genre relationships: {len(book_genres):,}")
print(f"Unique genres: {len(genres):,}")

year_genre_counts = (
    exploded
    .groupby(["pub_year", "genres_list"])
    .size()
    .reset_index(name="count")
)


print("Unique genres:", len(genres))
print("Book/genre relationships:", len(book_genres))

print("\nGenres:")
display(genres.head(20))

print("\nBook/Genre relationships:")
display(book_genres.head(10))

print("\nYear/Genre counts:")
display(year_genre_counts.head(10))

Book-genre relationships: 5,543,202
Unique genres: 1,440
Unique genres: 1440
Book/genre relationships: 5543202

Genres:


,genre_name
0,10th century
1,11th century
2,12th century
3,13th century
4,14th century
5,15th century
6,16th century
7,17th century
8,1864 shenandoah campaign
9,18th century



Book/Genre relationships:


,book_key,genre_name
0,615225.Sharpe_s_Devil,historical fiction
1,615225.Sharpe_s_Devil,historical
2,615225.Sharpe_s_Devil,war
3,615225.Sharpe_s_Devil,adventure
4,615225.Sharpe_s_Devil,military fiction
5,615225.Sharpe_s_Devil,audiobook
6,615225.Sharpe_s_Devil,action
7,615225.Sharpe_s_Devil,ebooks
8,615225.Sharpe_s_Devil,19th century
9,278794.Blood_Fever,young adult



Year/Genre counts:


,pub_year,genres_list,count
0,1000,ancient,5
1,1000,anglo saxon,5
2,1000,anthropology,3
3,1000,asia,12
4,1000,asian literature,6
5,1000,australia,1
6,1000,biography,5
7,1000,business,14
8,1000,christian,9
9,1000,christian fiction,5


### 8. Filter Publication Years

Publication years are restricted to books published between 1900 and 2016. This range removes invalid, missing, or extreme year values that could distort publication trend analysis while retaining the historical period represented by the dataset.

A copy of the filtered records is created in `valid`, and the record counts before and after filtering are displayed to verify the effect of the transformation.

In [ ]:
valid = df[
    (df["pub_year"] >= 1900) &
    (df["pub_year"] <= 2016)
].copy()

print("Total combined records:", len(df))
print("Records after year filter:", len(valid))

Total combined records: 4820937
Records after year filter: 4352106


### 9. Clean and Standardize Author Data

A custom `clean_authors()` function is used to standardize the author field. Because some records contain multiple authors stored as lists or arrays, the function converts these values into a consistent list format, removes unnecessary whitespace, and excludes blank author entries.

The cleaned results are stored in `authors_list` while preserving the original `author` column for reference. A sample of the original and cleaned values is displayed to verify the transformation.

In [18]:
def clean_authors(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return [
            " ".join(str(author).split()).strip()
            for author in value
            if str(author).strip()
        ]
    return []


valid["authors_list"] = valid["author"].apply(clean_authors)

# Inspect the results
display(valid[["author", "authors_list"]].head(10))

,author,authors_list
0,[Bernard Cornwell],[Bernard Cornwell]
1,[Charlie Higson],[Charlie Higson]
2,[Gosho Aoyama],[Gosho Aoyama]
3,[David McDonald],[David McDonald]
4,[Jerry Ahern],[Jerry Ahern]
5,[Louis L'Amour],[Louis L'Amour]
6,[Don Pendleton],[Don Pendleton]
7,[Lincoln Child],[Lincoln Child]
8,[Jinsei Kataoka],[Jinsei Kataoka]
9,[Sigmund Brouwer],[Sigmund Brouwer]


### 10. Create a Unique Book Identifier

A custom `create_book_key()` function generates a consistent identifier for each book. When an ISBN is available, it is used as the preferred identifier because it provides a standardized way to distinguish book editions.

For records without an ISBN, a fallback key is created using the book title, author information, and publication year. This approach allows books with missing ISBN values to remain identifiable within the dataset.

The resulting `book_key` is used throughout the relational database to connect books with their authors, genres, and cross-dataset matches. A sample of the generated keys is displayed to verify the results.

In [ ]:
def create_book_key(row):
    # Use ISBN when available
    if pd.notna(row["isbn_clean"]) and str(row["isbn_clean"]).strip():
        return f"isbn:{str(row['isbn_clean']).strip()}"
    
    # Otherwise use title + authors + publication year
    authors = "|".join(sorted(row["authors_list"]))
    
    return (
        f"title:{str(row['name']).strip().lower()}|"
        f"author:{authors.lower()}|"
        f"year:{row['pub_year']}"
    )


valid["book_key"] = valid.apply(create_book_key, axis=1)

display(
    valid[
        [
            "name",
            "authors_list",
            "pub_year",
            "isbn_clean",
            "book_key"
        ]
    ].head(10)
)

,name,authors_list,pub_year,isbn_clean,book_key
0,Sharpe's Devil,[Bernard Cornwell],1992,9780060932299,isbn:9780060932299
1,Blood Fever,[Charlie Higson],2006,9780786836628,isbn:9780786836628
2,Detektiv Conan vs. Kaito Kid,[Gosho Aoyama],2004,9783770476374,isbn:9783770476374
3,Marvel's Captain America: Sub Rosa,[David McDonald],2016,9781772752014,isbn:9781772752014
4,The Awakening,[Jerry Ahern],1984,9780821714782,isbn:9780821714782
5,Last of the Breed,[Louis L'Amour],1986,NaN,title:last of the breed|author:louis l'amour|y...
6,Terrible Tuesday,[Don Pendleton],1979,9781497687622,isbn:9781497687622
7,Terminal Freeze,[Lincoln Child],2008,9780385515511,isbn:9780385515511
8,"Eureka Seven: Psalms of Planets, Vol. 2",[Jinsei Kataoka],2005,9781594096914,isbn:9781594096914
9,Devil's Pass,[Sigmund Brouwer],2012,9781554699384,isbn:9781554699384


### 11. Validate Book Identifiers

The newly created `book_key` values are evaluated to determine how many valid records are present, how many unique books are represented, and how many duplicate records remain.

This validation helps identify duplicate book records before the data is used to build the relational database and ensures that the book identifier is functioning as intended.

In [21]:
print("Total valid records:", len(valid))
print("Unique book keys:", valid["book_key"].nunique())
print("Duplicate records:", valid["book_key"].duplicated().sum())

Total valid records: 4352106
Unique book keys: 1487805
Duplicate records: 2864301


### 12. Build Relational Database DataFrames

The cleaned dataset is transformed into the individual DataFrames required for the project's relational database. Separating the data into related tables reduces redundancy and allows books, authors, genres, and source datasets to be connected through defined relationships.

The `BOOKS` table contains one record per unique book. The `AUTHORS` table contains unique authors, while `BOOK_AUTHORS` establishes the many-to-many relationship between books and authors.

Similarly, `GENRES` contains the unique cleaned Goodreads genres, and `BOOK_GENRES` connects books to their genres. The `SOURCE_GENRES` table preserves the original genre classification from the source Parquet files, while `BOOK_SOURCE_GENRES` records which source dataset contained each book.

This structure provides the foundation for the SQLite relational database and allows the project to perform more advanced SQL queries without duplicating book, author, or genre information.

In [ ]:
# One row per unique book.

books = (
    valid[
        [
            "book_key",
            "name",
            "pub_year",
            "star_rating",
            "num_ratings",
            "isbn_clean"
        ]
    ]
    .drop_duplicates(subset="book_key")
    .reset_index(drop=True)
)

print("BOOKS:", len(books))

# Create one row per unique author.

authors = (
    valid[["authors_list"]]
    .explode("authors_list")
    .rename(columns={"authors_list": "author_name"})
)

# Remove missing/blank authors
authors = authors[
    authors["author_name"].notna() &
    (authors["author_name"] != "")
].copy()

# Get each author only once
authors = (
    authors[["author_name"]]
    .drop_duplicates()
    .sort_values("author_name")
    .reset_index(drop=True)
)

print("AUTHORS:", len(authors))


book_authors = (
    valid[
        [
            "book_key",
            "authors_list"
        ]
    ]
    .explode("authors_list")
    .rename(columns={"authors_list": "author_name"})
)

book_authors = book_authors[
    book_authors["author_name"].notna() &
    (book_authors["author_name"] != "")
].copy()

book_authors = (
    book_authors[
        [
            "book_key",
            "author_name"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("BOOK_AUTHORS relationships:", len(book_authors))



valid["genres_list"] = valid["genres"].apply(
    lambda g: [
        str(x).strip().lower()
        for x in g
    ]
    if isinstance(g, (list, tuple, np.ndarray))
    else []
)

# Explode so each book/genre combination becomes a row
exploded = valid.explode("genres_list")

# Remove missing/blank genres
exploded = exploded[
    exploded["genres_list"].notna() &
    (exploded["genres_list"] != "")
].copy()

# Remove broad labels that aren't useful for our
# specific genre analysis
fiction_labels = {
    "fiction",
    "non-fiction",
    "nonfiction"
}

exploded = exploded[
    ~exploded["genres_list"].isin(fiction_labels)
].copy()


genres = (
    exploded[["genres_list"]]
    .rename(columns={"genres_list": "genre_name"})
    .drop_duplicates()
    .sort_values("genre_name")
    .reset_index(drop=True)
)

print("GENRES:", len(genres))


book_genres = (
    exploded[
        [
            "book_key",
            "genres_list"
        ]
    ]
    .rename(columns={"genres_list": "genre_name"})
    .drop_duplicates()
    .reset_index(drop=True)
)

print("BOOK_GENRES relationships:", len(book_genres))


source_genres = (
    valid[["source_genre"]]
    .drop_duplicates()
    .sort_values("source_genre")
    .reset_index(drop=True)
)

print("SOURCE_GENRES:", len(source_genres))



book_source_genres = (
    valid[
        [
            "book_key",
            "source_genre"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "BOOK_SOURCE_GENRES relationships:",
    len(book_source_genres)
)

BOOKS: 1487805
AUTHORS: 482461
BOOK_AUTHORS relationships: 1488070
GENRES: 1433
BOOK_GENRES relationships: 4981671
SOURCE_GENRES: 100
BOOK_SOURCE_GENRES relationships: 4291574


### 13. Validate Relational DataFrames

A summary of all relational DataFrames is generated to verify that each table and relationship contains the expected number of records.

The counts include unique books, authors, and genres, along with the relationship records connecting books to authors, genres, and their original source datasets. This provides a final validation of the data structures before they are written to the SQLite database.

In [ ]:
print("\n========== DATABASE DATASET SUMMARY ==========")

print(f"Books:                 {len(books):,}")
print(f"Authors:               {len(authors):,}")
print(f"Book-Author links:     {len(book_authors):,}")
print(f"Genres:                {len(genres):,}")
print(f"Book-Genre links:      {len(book_genres):,}")
print(f"Source Genres:         {len(source_genres):,}")
print(f"Book-Source links:     {len(book_source_genres):,}")


========== DATABASE DATASET SUMMARY ==========
Books:                 1,487,805
Authors:               482,461
Book-Author links:     1,488,070
Genres:                1,433
Book-Genre links:      4,981,671
Source Genres:         100
Book-Source links:     4,291,574


### 14. Validate Book Relationships

The relationship DataFrames are checked to ensure that every referenced `book_key` exists in the `BOOKS` DataFrame.

This validation is performed for book-author, book-genre, and book-source relationships. Any relationship pointing to a book that does not exist in the primary `BOOKS` table would indicate a referential integrity problem that should be corrected before loading the data into SQLite.

In [ ]:
missing_book_authors = ~book_authors["book_key"].isin(
    books["book_key"]
)

missing_book_genres = ~book_genres["book_key"].isin(
    books["book_key"]
)

missing_book_sources = ~book_source_genres["book_key"].isin(
    books["book_key"]
)

print(
    "Book-author records with missing books:",
    missing_book_authors.sum()
)

print(
    "Book-genre records with missing books:",
    missing_book_genres.sum()
)

print(
    "Book-source records with missing books:",
    missing_book_sources.sum()
)

Book-author records with missing books: 0
Book-genre records with missing books: 0
Book-source records with missing books: 0


### 15. Inspect Relational DataFrames

The first records from each relational DataFrame are displayed to confirm that the tables contain the expected fields and cleaned values.

Inspecting the individual tables provides a final visual check of the primary `BOOKS`, `AUTHORS`, and `GENRES` tables, as well as the relationship tables connecting books to authors, genres, and their original source datasets. This helps identify formatting or data-cleaning issues before the DataFrames are loaded into the SQLite database.

In [25]:
print("\nBOOKS")
display(books.head())

print("\nAUTHORS")
display(authors.head(10))

print("\nBOOK_AUTHORS")
display(book_authors.head(10))

print("\nGENRES")
display(genres.head(20))

print("\nBOOK_GENRES")
display(book_genres.head(10))

print("\nSOURCE_GENRES")
display(source_genres.head(20))

print("\nBOOK_SOURCE_GENRES")
display(book_source_genres.head(10))


BOOKS


,book_key,name,pub_year,star_rating,num_ratings,isbn_clean
0,isbn:9780060932299,Sharpe's Devil,1992,4.14,8141,9780060932299
1,isbn:9780786836628,Blood Fever,2006,4.02,7516,9780786836628
2,isbn:9783770476374,Detektiv Conan vs. Kaito Kid,2004,4.33,231,9783770476374
3,isbn:9781772752014,Marvel's Captain America: Sub Rosa,2016,3.39,74,9781772752014
4,isbn:9780821714782,The Awakening,1984,3.87,305,9780821714782



AUTHORS


,author_name
0,!
1,"""Albert"""
2,"""Big"" John McCarthy"
3,"""J"""
4,"""Janosch"""
5,"""Laughing"" Larry Berger"
6,"""Lebanon"" Levi Stoltzfus"
7,"""Markku Oksanen, Veikko Launis & Seppo Sajama"
8,"""Miss Naomi"""
9,"""Panda"" Andy McAllister"



BOOK_AUTHORS


...

,book_key,genre_name
0,isbn:9780060932299,historical fiction
1,isbn:9780060932299,historical
2,isbn:9780060932299,war
3,isbn:9780060932299,adventure
4,isbn:9780060932299,military fiction
5,isbn:9780060932299,audiobook
6,isbn:9780060932299,action
7,isbn:9780060932299,ebooks
8,isbn:9780060932299,19th century
9,isbn:9780786836628,young adult



SOURCE_GENRES


,source_genre
0,action
1,adult
2,adventure
3,amazon
4,american_history
5,animals
6,anthologies
7,art
8,audiobook
9,bdsm



BOOK_SOURCE_GENRES


,book_key,source_genre
0,isbn:9780060932299,action
1,isbn:9780786836628,action
2,isbn:9783770476374,action
3,isbn:9781772752014,action
4,isbn:9780821714782,action
5,title:last of the breed|author:louis l'amour|y...,action
6,isbn:9781497687622,action
7,isbn:9780385515511,action
8,isbn:9781594096914,action
9,isbn:9781554699384,action


### 16. Load and Prepare the Second Dataset

The second dataset contains popular Goodreads **Fantasy** and **Science Fiction** books and will be integrated with the primary Goodreads dataset.

The Fantasy and Science Fiction CSV files are loaded separately, and a `source_genre` column is added to identify the original dataset for each record. The two datasets are then combined into a single DataFrame.

The original CSV index column, `Unnamed: 0`, is removed because it does not contain meaningful analytical information. Exact duplicate records are also removed, and the DataFrame index is reset.

The resulting `scifi_fantasy_books` DataFrame provides a clean, combined dataset that can later be matched against the relational Goodreads `BOOKS` table.

In [ ]:
from pathlib import Path
import pandas as pd

scifi_folder = Path("../Data/SciFi_Fantasy")


fan = pd.read_csv(
    scifi_folder / "goodreads_fan_books_clean.csv"
)

sf = pd.read_csv(
    scifi_folder / "goodreads_sf_books_clean.csv"
)


fan["source_genre"] = "Fantasy"
sf["source_genre"] = "Science Fiction"



scifi_fantasy_books = pd.concat(
    [fan, sf],
    ignore_index=True
)


scifi_fantasy_books = scifi_fantasy_books.drop(
    columns=["Unnamed: 0"]
)



scifi_fantasy_books = scifi_fantasy_books.drop_duplicates()


scifi_fantasy_books = scifi_fantasy_books.reset_index(
    drop=True
)


print("Sci-Fi/Fantasy dataset prepared successfully.")
print(f"Rows: {len(scifi_fantasy_books):,}")
print(f"Columns: {len(scifi_fantasy_books.columns)}")

display(scifi_fantasy_books.head())

Sci-Fi/Fantasy dataset prepared successfully.
Rows: 2,491
Columns: 9


,title,author,pub_year,avg_rate,num_rate,shelved,series_name,series_num,source_genre
0,Harry Potter and the Philosopher's Stone,J.K. Rowling,1997,4.47,8827238,71462,Harry Potter,1.0,Fantasy
1,Harry Potter and the Chamber of Secrets,J.K. Rowling,1998,4.43,3409926,59741,Harry Potter,2.0,Fantasy
2,Harry Potter and the Prisoner of Azkaban,J.K. Rowling,1999,4.58,3595393,59290,Harry Potter,3.0,Fantasy
3,The Hobbit,J.R.R. Tolkien,1937,4.28,3483329,58644,NaN,NaN,Fantasy
4,Harry Potter and the Goblet of Fire,J.K. Rowling,2000,4.57,3164528,57211,Harry Potter,4.0,Fantasy


### 17. Validate Second Dataset Composition

The number of records from each original source genre is counted to verify that the combined Sci-Fi/Fantasy dataset retained the expected Fantasy and Science Fiction classifications.

This check confirms that the `source_genre` field was assigned correctly and that both source datasets are represented in the combined DataFrame.

In [27]:
print("\nSource Genre Counts")
print("-" * 40)

print(
    scifi_fantasy_books["source_genre"].value_counts()
)


Source Genre Counts
----------------------------------------
source_genre
Fantasy            1246
Science Fiction    1245
Name: count, dtype: int64


### 18. Check for Missing Values

The combined Sci-Fi/Fantasy dataset is checked for missing values across all columns. This provides a quick assessment of data completeness and identifies fields that may require additional cleaning or consideration before the records are integrated with the primary Goodreads dataset.

In [28]:
print("\nMissing Values")
print("-" * 40)

display(
    scifi_fantasy_books.isna().sum()
)


Missing Values
----------------------------------------


title             0
author            0
pub_year          0
avg_rate          0
num_rate          0
shelved           0
series_name     654
series_num      655
source_genre      0
dtype: int64

### 19. Clean and Standardize the Second Dataset

The second dataset is cleaned and standardized to make it compatible with the primary Goodreads dataset and the relational database structure.

Column names are normalized by removing leading or trailing whitespace, converting names to lowercase, and replacing spaces with underscores. Numeric fields are converted to appropriate numeric data types, with invalid values converted to missing values.

Text fields such as book titles, authors, and series names are converted to a consistent string format and stripped of unnecessary whitespace. These transformations improve consistency and reduce potential matching issues when the two Goodreads datasets are compared and integrated.

In [ ]:
scifi_fantasy_books.columns = (
    scifi_fantasy_books.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


scifi_fantasy_books["pub_year"] = pd.to_numeric(
    scifi_fantasy_books["pub_year"],
    errors="coerce"
)

scifi_fantasy_books["avg_rate"] = pd.to_numeric(
    scifi_fantasy_books["avg_rate"],
    errors="coerce"
)

scifi_fantasy_books["num_rate"] = pd.to_numeric(
    scifi_fantasy_books["num_rate"],
    errors="coerce"
)

scifi_fantasy_books["shelved"] = pd.to_numeric(
    scifi_fantasy_books["shelved"],
    errors="coerce"
)

scifi_fantasy_books["series_num"] = pd.to_numeric(
    scifi_fantasy_books["series_num"],
    errors="coerce"
)


for column in ["title", "author", "series_name"]:
    scifi_fantasy_books[column] = (
        scifi_fantasy_books[column]
        .astype("string")
        .str.strip()
    )


print("Second dataset cleaned successfully.")

display(
    scifi_fantasy_books.head()
)

Second dataset cleaned successfully.


,title,author,pub_year,avg_rate,num_rate,shelved,series_name,series_num,source_genre
0,Harry Potter and the Philosopher's Stone,J.K. Rowling,1997,4.47,8827238,71462,Harry Potter,1.0,Fantasy
1,Harry Potter and the Chamber of Secrets,J.K. Rowling,1998,4.43,3409926,59741,Harry Potter,2.0,Fantasy
2,Harry Potter and the Prisoner of Azkaban,J.K. Rowling,1999,4.58,3595393,59290,Harry Potter,3.0,Fantasy
3,The Hobbit,J.R.R. Tolkien,1937,4.28,3483329,58644,<NA>,NaN,Fantasy
4,Harry Potter and the Goblet of Fire,J.K. Rowling,2000,4.57,3164528,57211,Harry Potter,4.0,Fantasy


### 20. Create a Matching Key for the Second Dataset

A normalized `match_key` is created for each Sci-Fi/Fantasy book using its title and author. Both fields are converted to lowercase and stripped of leading and trailing whitespace to improve consistency during matching.

This key provides a standardized way to compare records from the second dataset with books in the primary Goodreads dataset. The number of unique matching keys is also calculated to identify potential duplicate records before the integration step.

In [ ]:
def create_external_book_key(row):
    """
    Creates a normalized matching key using book title
    and author.
    """

    title = str(row["title"]).lower().strip()
    author = str(row["author"]).lower().strip()

    return f"{title}|{author}"


scifi_fantasy_books["match_key"] = (
    scifi_fantasy_books.apply(
        create_external_book_key,
        axis=1
    )
)


print(
    "Unique Sci-Fi/Fantasy matching keys:",
    scifi_fantasy_books["match_key"].nunique()
)

Unique Sci-Fi/Fantasy matching keys: 2351


### 21. Verify Primary Dataset Columns

The column names in the primary `BOOKS` DataFrame are displayed to verify the available fields before creating the cross-dataset matching logic.

This confirms that the expected book identifiers and metadata are available and helps ensure that the matching process uses the correct column names from the relational dataset.

In [31]:
print(books.columns.tolist())

['book_key', 'name', 'pub_year', 'star_rating', 'num_ratings', 'isbn_clean']


### 22. Normalize Titles and Authors for Matching

Two custom functions are used to normalize book titles and author names before attempting to match records between the two Goodreads datasets.

The normalization process converts text to lowercase, removes leading and trailing whitespace, and removes common punctuation that could cause otherwise identical records to appear different. Missing values are converted to empty strings.

The resulting `match_title` and `match_author` fields provide standardized values that can be combined into matching keys for cross-dataset record linkage.

In [ ]:
def normalize_title(value):
    """
    Creates a normalized title for cross-dataset matching.
    """

    if pd.isna(value):
        return ""

    return (
        str(value)
        .lower()
        .strip()
        .replace(":", "")
        .replace(",", "")
        .replace(".", "")
        .replace("'", "")
        .replace('"', "")
    )


def normalize_author(value):
    """
    Creates a normalized author name for cross-dataset matching.
    """

    if pd.isna(value):
        return ""

    return (
        str(value)
        .lower()
        .strip()
        .replace(".", "")
        .replace(",", "")
        .replace("'", "")
        .replace('"', "")
    )


scifi_fantasy_books["match_title"] = (
    scifi_fantasy_books["title"]
    .apply(normalize_title)
)

scifi_fantasy_books["match_author"] = (
    scifi_fantasy_books["author"]
    .apply(normalize_author)
)

### 23. Create the Primary Dataset Matching Table

A dedicated matching table is created from the primary relational dataset by combining the `BOOKS`, `BOOK_AUTHORS`, and `AUTHORS` DataFrames.

The table associates each book with its author while retaining the primary `book_key` and book title. The same title and author normalization functions used for the second dataset are then applied to create comparable `match_title` and `match_author` fields.

This creates a consistent structure that can be used to identify corresponding books across the two datasets.

In [ ]:
book_author_match = (
    book_authors
    .merge(
        books[["book_key", "name"]],
        on="book_key",
        how="inner"
    )
    .merge(
        authors[["author_name"]],
        on="author_name",
        how="inner"
    )
)

book_author_match["match_title"] = (
    book_author_match["name"]
    .apply(normalize_title)
)

book_author_match["match_author"] = (
    book_author_match["author_name"]
    .apply(normalize_author)
)


print("Rows:", len(book_author_match))
print("Columns:", book_author_match.columns.tolist())

display(book_author_match.head(10))

Rows: 1488070
Columns: ['book_key', 'author_name', 'name', 'match_title', 'match_author']


,book_key,author_name,name,match_title,match_author
0,isbn:9780060932299,Bernard Cornwell,Sharpe's Devil,sharpes devil,bernard cornwell
1,isbn:9780786836628,Charlie Higson,Blood Fever,blood fever,charlie higson
2,isbn:9783770476374,Gosho Aoyama,Detektiv Conan vs. Kaito Kid,detektiv conan vs kaito kid,gosho aoyama
3,isbn:9781772752014,David McDonald,Marvel's Captain America: Sub Rosa,marvels captain america sub rosa,david mcdonald
4,isbn:9780821714782,Jerry Ahern,The Awakening,the awakening,jerry ahern
5,title:last of the breed|author:louis l'amour|y...,Louis L'Amour,Last of the Breed,last of the breed,louis lamour
6,isbn:9781497687622,Don Pendleton,Terrible Tuesday,terrible tuesday,don pendleton
7,isbn:9780385515511,Lincoln Child,Terminal Freeze,terminal freeze,lincoln child
8,isbn:9781594096914,Jinsei Kataoka,"Eureka Seven: Psalms of Planets, Vol. 2",eureka seven psalms of planets vol 2,jinsei kataoka
9,isbn:9781554699384,Sigmund Brouwer,Devil's Pass,devils pass,sigmund brouwer


### 24. Match Primary Books to the Sci-Fi/Fantasy Dataset

The normalized title and author fields are used to identify books that appear in both the primary Goodreads dataset and the Sci-Fi/Fantasy dataset.

Each matched record is assigned the corresponding `book_key` from the primary relational dataset and the original row identifier from the Sci-Fi/Fantasy dataset. Only these two identifiers are retained in the final `book_dataset_matches` DataFrame because they are sufficient to establish the cross-dataset relationship.

Duplicate relationships are removed to ensure that each primary book and Sci-Fi/Fantasy record is represented only once. The total number of matched books is then reported and a sample of the relationships is displayed for verification.

In [ ]:
book_dataset_matches = (
    scifi_fantasy_books[
        ["match_title", "match_author"]
    ]
    .reset_index()
    .rename(
        columns={"index": "scifi_fantasy_book_id"}
    )
    .merge(
        book_author_match[
            [
                "book_key",
                "match_title",
                "match_author"
            ]
        ],
        on=["match_title", "match_author"],
        how="inner"
    )
)


book_dataset_matches = book_dataset_matches[
    [
        "book_key",
        "scifi_fantasy_book_id"
    ]
].drop_duplicates()


print(
    "Matched books:",
    len(book_dataset_matches)
)

display(book_dataset_matches.head(20))

Matched books: 1940


,book_key,scifi_fantasy_book_id
0,title:harry potter and the chamber of secrets|...,1
1,isbn:9780439655484,2
2,title:harry potter and the goblet of fire|auth...,4
3,title:harry potter and the half-blood prince|a...,5
4,isbn:9780439358064,6
5,isbn:9781855496644,6
6,isbn:9780553588484,7
7,title:the fellowship of the ring|author:j.r.r....,8
8,title:a clash of kings|author:george r.r. mart...,9
9,isbn:9780756404079,10


### 25. Validate Cross-Dataset Matches

The cross-dataset relationships are validated by comparing the number of records in the Sci-Fi/Fantasy dataset with the number of unique matched records from both datasets.

The validation checks for duplicate primary book identifiers, duplicate Sci-Fi/Fantasy identifiers, and missing identifiers. These checks help confirm that the matching process produced valid, unique relationships without incomplete references.

This step provides confidence that `book_dataset_matches` is ready to be incorporated into the relational database.

In [ ]:
print("=" * 60)
print("CROSS-DATASET MATCH VALIDATION")
print("=" * 60)

print(
    f"Sci-Fi/Fantasy records: "
    f"{len(scifi_fantasy_books):,}"
)

print(
    f"Matched primary books: "
    f"{book_dataset_matches['book_key'].nunique():,}"
)

print(
    f"Matched Sci-Fi/Fantasy records: "
    f"{book_dataset_matches['scifi_fantasy_book_id'].nunique():,}"
)

print(
    "Duplicate book IDs:",
    book_dataset_matches["book_key"].duplicated().sum()
)

print(
    "Duplicate Sci-Fi/Fantasy IDs:",
    book_dataset_matches[
        "scifi_fantasy_book_id"
    ].duplicated().sum()
)

print(
    "Missing book IDs:",
    book_dataset_matches["book_key"].isna().sum()
)

print(
    "Missing Sci-Fi/Fantasy IDs:",
    book_dataset_matches[
        "scifi_fantasy_book_id"
    ].isna().sum()
)

CROSS-DATASET MATCH VALIDATION
Sci-Fi/Fantasy records: 2,491
Matched primary books: 1,822
Matched Sci-Fi/Fantasy records: 1,903
Duplicate book IDs: 118
Duplicate Sci-Fi/Fantasy IDs: 37
Missing book IDs: 0
Missing Sci-Fi/Fantasy IDs: 0


### 26. Check for Duplicate Primary Book Relationships

The number of duplicated `book_key` values in the cross-dataset matching table is checked to determine whether any primary Goodreads book is linked to multiple Sci-Fi/Fantasy records.

A result of zero indicates that each matched primary book has a single cross-dataset relationship, supporting the integrity of the matching table.

In [47]:
book_dataset_matches["book_key"].duplicated().sum()

np.int64(118)

In [48]:
book_dataset_matches["scifi_fantasy_book_id"].duplicated().sum()

np.int64(37)

### 27. Check for Duplicate Sci-Fi/Fantasy Relationships

The number of duplicated `scifi_fantasy_book_id` values is checked to determine whether any Sci-Fi/Fantasy record was matched to multiple primary Goodreads books.

A result of zero indicates that each matched Sci-Fi/Fantasy record corresponds to only one primary Goodreads book, helping confirm that the cross-dataset relationships are unique.

In [ ]:
duplicate_book_ids = (
    book_dataset_matches[
        book_dataset_matches["book_key"].duplicated(keep=False)
    ]
    .sort_values("book_key")
)

display(duplicate_book_ids.head(30))

,book_key,scifi_fantasy_book_id
299,isbn:9780007442911,353
1010,isbn:9780007442911,1305
439,isbn:9780007524273,538
1027,isbn:9780007524273,1331
31,isbn:9780060557812,34
1404,isbn:9780060557812,1812
26,isbn:9780060853976,29
1467,isbn:9780060853976,1893
42,isbn:9780060855925,47
1599,isbn:9780060855925,2060


### 28. Inspect Duplicate Sci-Fi/Fantasy Matches

Any Sci-Fi/Fantasy records that appear more than once in the cross-dataset matching table are isolated and sorted by `scifi_fantasy_book_id`.

The resulting DataFrame is displayed to allow the duplicate relationships to be reviewed individually. If no records are returned, this confirms that no Sci-Fi/Fantasy book was matched to multiple primary Goodreads books.

In [51]:
duplicate_scifi_ids = (
    book_dataset_matches[
        book_dataset_matches["scifi_fantasy_book_id"].duplicated(
            keep=False
        )
    ]
    .sort_values("scifi_fantasy_book_id")
)

display(duplicate_scifi_ids.head(30))

,book_key,scifi_fantasy_book_id
4,isbn:9780439358064,6
5,isbn:9781855496644,6
26,isbn:9780060853976,29
27,title:good omens: the nice and accurate prophe...,29
57,isbn:9780375840401,64
58,isbn:9781400098644,64
59,isbn:9780307284532,64
65,isbn:9780060530921,71
66,isbn:9788888893754,71
98,isbn:9781561798421,112


### 29. Inspect Matched Records

A sample of matched records is reviewed by joining the cross-dataset relationship table back to both source datasets.

The primary Goodreads book information is combined with the corresponding Sci-Fi/Fantasy title, author, publication year, ratings, number of ratings, and source genre. This side-by-side comparison provides a final validation that the matching process is correctly identifying the same books across the two datasets.

The sample also allows differences in ratings and other metadata between the two Goodreads sources to be observed before the relationships are stored in the database.

In [ ]:
sample_matches = (
    book_dataset_matches
    .drop_duplicates()
    .head(20)
    .merge(
        books[
            [
                "book_key",
                "name",
                "pub_year",
                "star_rating",
                "num_ratings"
            ]
        ],
        on="book_key",
        how="left"
    )
)

scifi_lookup = (
    scifi_fantasy_books
    .reset_index()
    .rename(columns={"index": "scifi_fantasy_book_id"})
)

sample_matches = sample_matches.merge(
    scifi_lookup[
        [
            "scifi_fantasy_book_id",
            "title",
            "author",
            "pub_year",
            "avg_rate",
            "num_rate",
            "source_genre"
        ]
    ],
    on="scifi_fantasy_book_id",
    how="left",
    suffixes=("_goodreads", "_scifi_fantasy")
)

display(sample_matches)

,book_key,scifi_fantasy_book_id,name,pub_year_goodreads,star_rating,num_ratings,title,author,pub_year_scifi_fantasy,avg_rate,num_rate,source_genre
0,title:harry potter and the chamber of secrets|...,1,Harry Potter and the Chamber of Secrets,1998,4.43,4500160,Harry Potter and the Chamber of Secrets,J.K. Rowling,1998,4.43,3409926,Fantasy
1,isbn:9780439655484,2,Harry Potter and the Prisoner of Azkaban,1999,4.58,4849237,Harry Potter and the Prisoner of Azkaban,J.K. Rowling,1999,4.58,3595393,Fantasy
2,title:harry potter and the goblet of fire|auth...,4,Harry Potter and the Goblet of Fire,2000,4.57,4198921,Harry Potter and the Goblet of Fire,J.K. Rowling,2000,4.57,3164528,Fantasy
3,title:harry potter and the half-blood prince|a...,5,Harry Potter and the Half-Blood Prince,2005,4.58,3661172,Harry Potter and the Half-Blood Prince,J.K. Rowling,2005,4.58,2923256,Fantasy
4,isbn:9780439358064,6,Harry Potter and the Order of the Phoenix,2003,4.50,3799643,Harry Potter and the Order of the Phoenix,J.K. Rowling,2003,4.50,3019296,Fantasy
5,isbn:9781855496644,6,Harry Potter and the Order of the Phoenix,2003,4.60,305052,Harry Potter and the Order of the Phoenix,J.K. Rowling,2003,4.50,3019296,Fantasy
6,isbn:9780553588484,7,A Game of Thrones,1996,4.45,2736949,A Game of Thrones,George R.R. Martin,1996,4.44,2281001,Fantasy
7,title:the fellowship of the ring|author:j.r.r....,8,The Fellowship of the Ring,1954,4.41,3154997,The Fellowship of the Ring,J.R.R. Tolkien,1954,4.38,2617415,Fantasy
8,title:a clash of kings|author:george r.r. mart...,9,A Clash of Kings,1998,4.42,1021673,A Clash of Kings,George R.R. Martin,1998,4.41,865168,Fantasy
9,isbn:9780756404079,10,The Name of the Wind,2007,4.52,1083669,The Name of the Wind,Patrick Rothfuss,2007,4.52,854536,Fantasy


### 30. Assign Primary Keys to the Sci-Fi/Fantasy Dataset

A unique primary key, `scifi_fantasy_book_id`, is assigned to every record in the Sci-Fi/Fantasy dataset.

The DataFrame index is first reset to ensure a consistent sequential order, and the new identifier is inserted as the first column. This identifier provides a stable reference for each Sci-Fi/Fantasy book and allows the dataset to be linked to the primary Goodreads `BOOKS` table through the cross-dataset relationship table.

The total number of Sci-Fi/Fantasy records is then reported and the resulting structure is inspected.

In [ ]:
scifi_fantasy_books = scifi_fantasy_books.reset_index(drop=True)

scifi_fantasy_books.insert(
    0,
    "scifi_fantasy_book_id",
    range(1, len(scifi_fantasy_books) + 1)
)


print(
    "Sci-Fi/Fantasy records:",
    len(scifi_fantasy_books)
)

display(
    scifi_fantasy_books.head()
)

Sci-Fi/Fantasy records: 2491


,scifi_fantasy_book_id,title,author,pub_year,avg_rate,num_rate,shelved,series_name,series_num,source_genre,match_key,match_title,match_author
0,1,Harry Potter and the Philosopher's Stone,J.K. Rowling,1997,4.47,8827238,71462,Harry Potter,1.0,Fantasy,harry potter and the philosopher's stone|j.k. ...,harry potter and the philosophers stone,jk rowling
1,2,Harry Potter and the Chamber of Secrets,J.K. Rowling,1998,4.43,3409926,59741,Harry Potter,2.0,Fantasy,harry potter and the chamber of secrets|j.k. r...,harry potter and the chamber of secrets,jk rowling
2,3,Harry Potter and the Prisoner of Azkaban,J.K. Rowling,1999,4.58,3595393,59290,Harry Potter,3.0,Fantasy,harry potter and the prisoner of azkaban|j.k. ...,harry potter and the prisoner of azkaban,jk rowling
3,4,The Hobbit,J.R.R. Tolkien,1937,4.28,3483329,58644,<NA>,NaN,Fantasy,the hobbit|j.r.r. tolkien,the hobbit,jrr tolkien
4,5,Harry Potter and the Goblet of Fire,J.K. Rowling,2000,4.57,3164528,57211,Harry Potter,4.0,Fantasy,harry potter and the goblet of fire|j.k. rowling,harry potter and the goblet of fire,jk rowling


### 31. Rebuild the Cross-Dataset Match Table

After assigning permanent primary keys to the Sci-Fi/Fantasy dataset, the cross-dataset relationship table is rebuilt using the new `scifi_fantasy_book_id` values.

The normalized title and author fields are used to reconnect each Sci-Fi/Fantasy record with its corresponding primary Goodreads book. Only the two primary identifiers are retained in the final relationship table, creating a clean link between the datasets.

Duplicate relationships are removed and the index is reset. The resulting table is then inspected to verify that the cross-dataset relationships were successfully rebuilt using the new identifiers.

In [ ]:
scifi_lookup = scifi_fantasy_books[
    [
        "scifi_fantasy_book_id",
        "match_title",
        "match_author"
    ]
]


book_dataset_matches = (
    scifi_lookup
    .merge(
        book_author_match[
            [
                "book_key",
                "match_title",
                "match_author"
            ]
        ],
        on=["match_title", "match_author"],
        how="inner"
    )
)


book_dataset_matches = (
    book_dataset_matches[
        [
            "book_key",
            "scifi_fantasy_book_id"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print(
    "Cross-dataset relationships:",
    len(book_dataset_matches)
)

display(
    book_dataset_matches.head(20)
)

Cross-dataset relationships: 1940


,book_key,scifi_fantasy_book_id
0,title:harry potter and the chamber of secrets|...,2
1,isbn:9780439655484,3
2,title:harry potter and the goblet of fire|auth...,5
3,title:harry potter and the half-blood prince|a...,6
4,isbn:9780439358064,7
5,isbn:9781855496644,7
6,isbn:9780553588484,8
7,title:the fellowship of the ring|author:j.r.r....,9
8,title:a clash of kings|author:george r.r. mart...,10
9,isbn:9780756404079,11


### 32. Validate Cross-Dataset Relationships

The rebuilt cross-dataset relationship table is validated to confirm that the identifiers and relationships remain unique after assigning permanent primary keys to the Sci-Fi/Fantasy dataset.

The validation reports the total number of Sci-Fi/Fantasy books, the number of unique matched Goodreads books, and the total number of cross-dataset relationships. It also checks for duplicate `book_key` and `scifi_fantasy_book_id` values.

These checks confirm that each matched Sci-Fi/Fantasy book is linked to a single Goodreads book and that the relationship table contains no unintended duplicate relationships.

In [ ]:
print("=" * 60)
print("CROSS-DATASET RELATIONSHIP VALIDATION")
print("=" * 60)

print(
    f"Sci-Fi/Fantasy books: "
    f"{scifi_fantasy_books['scifi_fantasy_book_id'].nunique():,}"
)

print(
    f"Matched Goodreads records: "
    f"{book_dataset_matches['book_key'].nunique():,}"
)

print(
    f"Cross-dataset relationships: "
    f"{len(book_dataset_matches):,}"
)

print(
    f"Books with multiple Goodreads matches: "
    f"{book_dataset_matches['book_key'].duplicated().sum():,}"
)

print(
    f"Sci-Fi/Fantasy books with multiple Goodreads matches: "
    f"{book_dataset_matches['scifi_fantasy_book_id'].duplicated().sum():,}"
)

CROSS-DATASET RELATIONSHIP VALIDATION
Sci-Fi/Fantasy books: 2,491
Matched Goodreads records: 1,822
Cross-dataset relationships: 1,940
Books with multiple Goodreads matches: 118
Sci-Fi/Fantasy books with multiple Goodreads matches: 37


### 33. Check for Duplicate Relationships

The complete cross-dataset relationship table is checked for exact duplicate rows.

A result of zero confirms that each `book_key` and `scifi_fantasy_book_id` relationship appears only once, providing an additional validation that the matching table is clean and ready for database integration.

In [62]:
print(
    "Duplicate relationships:",
    book_dataset_matches.duplicated().sum()
)

Duplicate relationships: 0


### 34. Save Sci-Fi/Fantasy Integration Tables

The prepared Sci-Fi/Fantasy dataset and the validated cross-dataset relationship table are saved as Parquet files in the `Data/prepared` directory.

Saving these tables separately preserves the cleaned and integrated data for use during the database construction phase without requiring the preparation and matching process to be repeated.

The saved files are:

- `scifi_fantasy_books.parquet` — the cleaned Sci-Fi/Fantasy dataset with assigned primary keys.
- `book_dataset_matches.parquet` — the validated relationships linking Sci-Fi/Fantasy books to the primary Goodreads `BOOKS` table.

In [ ]:
prepared_folder = Path("../Data/prepared")
prepared_folder.mkdir(parents=True, exist_ok=True)


scifi_fantasy_books.to_parquet(
    prepared_folder / "scifi_fantasy_books.parquet",
    index=False
)

book_dataset_matches.to_parquet(
    prepared_folder / "book_dataset_matches.parquet",
    index=False
)


print("Sci-Fi/Fantasy integration tables saved successfully.")

Sci-Fi/Fantasy integration tables saved successfully.


### 35. Verify Saved Integration Tables

The `Data/prepared` directory is checked to confirm that the Parquet files created during the integration process were successfully saved.

Each prepared file is listed so the final data preparation outputs can be verified before moving on to database construction and SQL analysis.

In [ ]:
print("Prepared files:")

for file in sorted(prepared_folder.glob("*.parquet")):
    print(f"  ✓ {file.name}")

Prepared files:
  ✓ authors.parquet
  ✓ book_authors.parquet
  ✓ book_dataset_matches.parquet
  ✓ book_genres.parquet
  ✓ book_source_genres.parquet
  ✓ books.parquet
  ✓ genres.parquet
  ✓ scifi_fantasy_books.parquet
  ✓ source_genres.parquet


### 36. Remove Temporary Matching Columns

The temporary `match_title` and `match_author` columns used during cross-dataset record matching are removed from the Sci-Fi/Fantasy dataset.

These fields were necessary for the integration process but are not part of the final dataset structure. Removing them keeps the prepared dataset focused on its actual book and metadata fields.

The remaining columns are displayed to verify the final structure.

In [ ]:
scifi_fantasy_books = scifi_fantasy_books.drop(
    columns=["match_title", "match_author"]
)


print("Final Sci-Fi/Fantasy columns:")
print(scifi_fantasy_books.columns.tolist())

Final Sci-Fi/Fantasy columns:
['scifi_fantasy_book_id', 'title', 'author', 'pub_year', 'avg_rate', 'num_rate', 'shelved', 'series_name', 'series_num', 'source_genre', 'match_key']


### 37. Re-save the Cleaned Sci-Fi/Fantasy Integration Table

The Sci-Fi/Fantasy integration table is saved again after removing the temporary matching columns.

This overwrites the previous Parquet file with the finalized version of the dataset, ensuring that the saved file contains only the fields required for database integration and analysis.

In [ ]:
scifi_fantasy_books.to_parquet(
    prepared_folder / "scifi_fantasy_books.parquet",
    index=False
)

print("Cleaned Sci-Fi/Fantasy dataset saved.")

Cleaned Sci-Fi/Fantasy dataset saved.


### 38. Assign Database Primary Keys

Database primary keys are assigned to the primary relational tables before they are loaded into SQLite.

Sequential IDs are created for the `BOOKS`, `AUTHORS`, `GENRES`, and `SOURCE_GENRES` tables. These identifiers provide stable primary keys that can be referenced by the relationship tables and enforce the structure of the relational database.

Lookup dictionaries are then created to translate the natural identifiers and descriptive values into their corresponding database IDs. These lookups will be used to construct the foreign-key relationships between the tables during database loading.

The number of IDs generated for each table is printed as a validation check.

In [ ]:
books = books.reset_index(drop=True)
books.insert(0, "book_id", books.index + 1)



authors = authors.reset_index(drop=True)
authors.insert(0, "author_id", authors.index + 1)


genres = genres.reset_index(drop=True)
genres.insert(0, "genre_id", genres.index + 1)


source_genres = source_genres.reset_index(drop=True)
source_genres.insert(0, "source_genre_id", source_genres.index + 1)


book_id_lookup = dict(
    zip(
        books["book_key"],
        books["book_id"]
    )
)

author_id_lookup = dict(
    zip(
        authors["author_name"],
        authors["author_id"]
    )
)

genre_id_lookup = dict(
    zip(
        genres["genre_name"],
        genres["genre_id"]
    )
)

source_genre_id_lookup = dict(
    zip(
        source_genres["source_genre"],
        source_genres["source_genre_id"]
    )
)


print("Book IDs:", len(book_id_lookup))
print("Author IDs:", len(author_id_lookup))
print("Genre IDs:", len(genre_id_lookup))
print("Source Genre IDs:", len(source_genre_id_lookup))

Book IDs: 1487805
Author IDs: 482461
Genre IDs: 1433
Source Genre IDs: 100


### 39. Convert Relationship Tables to Foreign Keys

The relationship DataFrames are converted from natural identifiers to database foreign keys using the lookup dictionaries created in the previous step.

For `BOOK_AUTHORS`, each `book_key` and `author_name` is mapped to its corresponding `book_id` and `author_id`. Similarly, `BOOK_GENRES` is converted to `book_id` and `genre_id`, while `BOOK_SOURCE_GENRES` is converted to `book_id` and `source_genre_id`.

Only the foreign-key columns required by the relational database are retained, and duplicate relationships are removed. The number of relationships in each table is then reported as a validation check.

In [ ]:
book_authors["book_id"] = book_authors["book_key"].map(
    book_id_lookup
)

book_authors["author_id"] = book_authors["author_name"].map(
    author_id_lookup
)

book_authors = book_authors[
    ["book_id", "author_id"]
].drop_duplicates()



book_genres["book_id"] = book_genres["book_key"].map(
    book_id_lookup
)

book_genres["genre_id"] = book_genres["genre_name"].map(
    genre_id_lookup
)

book_genres = book_genres[
    ["book_id", "genre_id"]
].drop_duplicates()


book_source_genres["book_id"] = book_source_genres["book_key"].map(
    book_id_lookup
)

book_source_genres["source_genre_id"] = (
    book_source_genres["source_genre"].map(
        source_genre_id_lookup
    )
)

book_source_genres = book_source_genres[
    ["book_id", "source_genre_id"]
].drop_duplicates()


print("BOOK_AUTHORS:", len(book_authors))
print("BOOK_GENRES:", len(book_genres))
print("BOOK_SOURCE_GENRES:", len(book_source_genres))

BOOK_AUTHORS: 1488070
BOOK_GENRES: 4981671
BOOK_SOURCE_GENRES: 4291574


### 40. Final Data Preparation Validation

A final validation is performed before the prepared DataFrames are loaded into the SQLite database.

The summary reports the number of records in each entity and relationship table. Foreign-key validation then checks for missing IDs in each relationship table to ensure that every relationship can be connected to a valid parent record.

The primary keys for the `BOOKS`, `AUTHORS`, `GENRES`, and `SOURCE_GENRES` tables are also checked for duplicates. A successful validation should show zero missing foreign keys and zero duplicate primary keys.

This confirms that the prepared relational data is structurally consistent and ready for database construction.

In [ ]:
print("=" * 60)
print("FINAL DATA PREPARATION VALIDATION")
print("=" * 60)

print(f"Books:                 {len(books):,}")
print(f"Authors:               {len(authors):,}")
print(f"Book-Author links:     {len(book_authors):,}")
print(f"Genres:                {len(genres):,}")
print(f"Book-Genre links:      {len(book_genres):,}")
print(f"Source Genres:         {len(source_genres):,}")
print(f"Book-Source links:     {len(book_source_genres):,}")

print("\nForeign Key Validation")
print("-" * 60)

print(
    "BOOK_AUTHORS missing book IDs:",
    book_authors["book_id"].isna().sum()
)

print(
    "BOOK_AUTHORS missing author IDs:",
    book_authors["author_id"].isna().sum()
)

print(
    "BOOK_GENRES missing book IDs:",
    book_genres["book_id"].isna().sum()
)

print(
    "BOOK_GENRES missing genre IDs:",
    book_genres["genre_id"].isna().sum()
)

print(
    "BOOK_SOURCE_GENRES missing book IDs:",
    book_source_genres["book_id"].isna().sum()
)

print(
    "BOOK_SOURCE_GENRES missing source genre IDs:",
    book_source_genres["source_genre_id"].isna().sum()
)

print("\nDuplicate Primary Keys")
print("-" * 60)

print("Duplicate book IDs:",
      books["book_id"].duplicated().sum())

print("Duplicate author IDs:",
      authors["author_id"].duplicated().sum())

print("Duplicate genre IDs:",
      genres["genre_id"].duplicated().sum())

print("Duplicate source genre IDs:",
      source_genres["source_genre_id"].duplicated().sum())

FINAL DATA PREPARATION VALIDATION
Books:                 1,487,805
Authors:               482,461
Book-Author links:     1,488,070
Genres:                1,433
Book-Genre links:      4,981,671
Source Genres:         100
Book-Source links:     4,291,574

Foreign Key Validation
------------------------------------------------------------
BOOK_AUTHORS missing book IDs: 0
BOOK_AUTHORS missing author IDs: 0
BOOK_GENRES missing book IDs: 0
BOOK_GENRES missing genre IDs: 0
BOOK_SOURCE_GENRES missing book IDs: 0
BOOK_SOURCE_GENRES missing source genre IDs: 0

Duplicate Primary Keys
------------------------------------------------------------
Duplicate book IDs: 0
Duplicate author IDs: 0
Duplicate genre IDs: 0
Duplicate source genre IDs: 0


### 41. Save Final Prepared Database Tables

The finalized relational DataFrames are saved as individual Parquet files in the `Data/prepared` directory.

Each file represents a table in the planned SQLite database:

- `books.parquet`
- `authors.parquet`
- `book_authors.parquet`
- `genres.parquet`
- `book_genres.parquet`
- `source_genres.parquet`
- `book_source_genres.parquet`

Separating the tables into individual files preserves the relational structure established during data preparation and provides the inputs needed for the database construction notebook.

In [70]:
prepared_folder = Path("../Data/prepared")
prepared_folder.mkdir(parents=True, exist_ok=True)

books.to_parquet(
    prepared_folder / "books.parquet",
    index=False
)

authors.to_parquet(
    prepared_folder / "authors.parquet",
    index=False
)

book_authors.to_parquet(
    prepared_folder / "book_authors.parquet",
    index=False
)

genres.to_parquet(
    prepared_folder / "genres.parquet",
    index=False
)

book_genres.to_parquet(
    prepared_folder / "book_genres.parquet",
    index=False
)

source_genres.to_parquet(
    prepared_folder / "source_genres.parquet",
    index=False
)

book_source_genres.to_parquet(
    prepared_folder / "book_source_genres.parquet",
    index=False
)

print("Prepared datasets saved successfully.")

Prepared datasets saved successfully.
